# 10_timeline_riesgos.ipynb

**Actividad de Grado MIA UC**  
**Sistema Inteligente de Vigilancia de Riesgos Documentales basado en RAG y LLMs**

Este notebook construye una versión corregida del **timeline de riesgos documentales** a partir de los riesgos extraídos y evaluados manualmente.

## Objetivo
Transformar la tabla de riesgos extraídos en una vista temporal y priorizada, infiriendo la fecha desde el documento fuente cuando el nombre del archivo contiene patrones como `YYYYMMDD`, `YYYY-MM-DD`, `DD-MM-YYYY` o códigos con año.

## Entradas
- `riesgos_evaluation_completed.xlsx` recomendado
- Alternativamente: `riesgos_extraidos.xlsx`

## Salidas
- `timeline_riesgos.csv`
- `timeline_riesgos.xlsx`
- `radar_riesgos_resumen.csv`
- `riesgos_priorizados.csv`
- `riesgos_priorizados.xlsx`
- `timeline_riesgos_completar_fechas.xlsx`


In [ ]:
# =====================================================
# 10_timeline_riesgos.ipynb
# Actividad de Grado MIA UC
# Patricia Patiño
# Versión corregida: inferencia de fechas YYYYMMDD
# =====================================================

!pip -q install pandas openpyxl matplotlib plotly

## 1. Importar librerías

In [ ]:
import re
import pandas as pd
import matplotlib.pyplot as plt
import plotly.express as px
from pathlib import Path
from google.colab import files

pd.set_option('display.max_colwidth', 160)

print('Librerías cargadas correctamente')

## 2. Cargar archivo de riesgos

Sube preferiblemente:

`riesgos_evaluation_completed.xlsx`

Si no lo tienes, puedes subir:

`riesgos_extraidos.xlsx`

In [ ]:
print('Sube riesgos_evaluation_completed.xlsx o riesgos_extraidos.xlsx')
uploaded = files.upload()
print('\nArchivos cargados:', list(uploaded.keys()))

## 3. Leer archivo

In [ ]:
possible_files = [
    'riesgos_evaluation_completed.xlsx',
    'riesgos_extraidos.xlsx'
]

input_path = None
for file in possible_files:
    if Path(file).exists():
        input_path = Path(file)
        break

if input_path is None:
    raise FileNotFoundError('No se encontró archivo de riesgos. Sube riesgos_evaluation_completed.xlsx o riesgos_extraidos.xlsx')

riesgos_df = pd.read_excel(input_path)

print('Archivo leído:', input_path)
print('Filas:', len(riesgos_df))
display(riesgos_df.head())
print('Columnas disponibles:')
print(riesgos_df.columns.tolist())

## 4. Filtrar riesgos válidos

In [ ]:
if 'riesgo_valido_manual' in riesgos_df.columns:
    riesgos_df['riesgo_valido_manual'] = pd.to_numeric(riesgos_df['riesgo_valido_manual'], errors='coerce')
    riesgos_validos = riesgos_df[riesgos_df['riesgo_valido_manual'] == 1].copy()
    print('Se filtraron riesgos válidos manualmente')
else:
    riesgos_validos = riesgos_df.copy()
    print('No existe columna riesgo_valido_manual. Se usan todos los riesgos.')

print('Riesgos válidos:', len(riesgos_validos))
display(riesgos_validos.head())

## 5. Normalizar columnas clave

In [ ]:
rename_map = {}

if 'source_filename' not in riesgos_validos.columns and 'filename' in riesgos_validos.columns:
    rename_map['filename'] = 'source_filename'
if 'source_doc_id' not in riesgos_validos.columns and 'doc_id' in riesgos_validos.columns:
    rename_map['doc_id'] = 'source_doc_id'
if 'source_chunk_id' not in riesgos_validos.columns and 'chunk_id' in riesgos_validos.columns:
    rename_map['chunk_id'] = 'source_chunk_id'
if 'source_page' not in riesgos_validos.columns and 'page' in riesgos_validos.columns:
    rename_map['page'] = 'source_page'

riesgos_validos = riesgos_validos.rename(columns=rename_map)

required = ['risk_id', 'risk_name', 'risk_category', 'severity', 'probability', 'source_filename']
missing = [c for c in required if c not in riesgos_validos.columns]

if missing:
    raise ValueError(f'Faltan columnas necesarias para timeline: {missing}')

# Crear columnas opcionales si no existen
for col in ['risk_description', 'evidence', 'recommended_action', 'source_doc_id', 'source_page', 'source_chunk_id']:
    if col not in riesgos_validos.columns:
        riesgos_validos[col] = ''

display(riesgos_validos[required].head())

## 6. Inferir fecha del documento fuente

Esta versión reconoce varios formatos comunes en nombres de documentos:

- `YYYYMMDD`, por ejemplo `20250829`
- `YYYY-MM-DD`, `YYYY_MM_DD`, `YYYY.MM.DD`
- `DD-MM-YYYY`, `DD_MM_YYYY`, `DD.MM.YYYY`
- `YYYYMM`, por ejemplo `202508`, asignando día 1
- códigos con año de dos dígitos, por ejemplo `CNS-0570-24`, asignando `2024-01-01`

La fecha inferida se usa como aproximación documental para construir el timeline.

In [ ]:
def inferir_fecha_desde_nombre(filename):
    text = str(filename)

    # 1. Formato YYYYMMDD, ejemplo 20250829
    match = re.search(r'(20\d{2})(\d{2})(\d{2})', text)
    if match:
        y, m, d = match.groups()
        fecha = pd.to_datetime(f'{y}-{m}-{d}', errors='coerce')
        if pd.notna(fecha):
            return fecha

    # 2. Formato YYYY-MM-DD / YYYY_MM_DD / YYYY.MM.DD
    match = re.search(r'(20\d{2})[-_\.](\d{1,2})[-_\.](\d{1,2})', text)
    if match:
        y, m, d = match.groups()
        fecha = pd.to_datetime(f'{y}-{m}-{d}', errors='coerce')
        if pd.notna(fecha):
            return fecha

    # 3. Formato DD-MM-YYYY / DD_MM_YYYY / DD.MM.YYYY
    match = re.search(r'(\d{1,2})[-_\.](\d{1,2})[-_\.](20\d{2})', text)
    if match:
        d, m, y = match.groups()
        fecha = pd.to_datetime(f'{y}-{m}-{d}', errors='coerce')
        if pd.notna(fecha):
            return fecha

    # 4. Formato YYYYMM, ejemplo 202508. Se asigna día 1.
    match = re.search(r'(20\d{2})(\d{2})(?!\d)', text)
    if match:
        y, m = match.groups()
        fecha = pd.to_datetime(f'{y}-{m}-01', errors='coerce')
        if pd.notna(fecha):
            return fecha

    # 5. Código con año de dos dígitos tipo CNS-0570-24. Se asigna 1 de enero.
    match = re.search(r'[-_](\d{2})(?:\D|$)', text)
    if match:
        yy = int(match.group(1))
        if 20 <= yy <= 35:
            fecha = pd.to_datetime(f'20{yy}-01-01', errors='coerce')
            if pd.notna(fecha):
                return fecha

    return pd.NaT


riesgos_validos['fecha_documento'] = riesgos_validos['source_filename'].apply(inferir_fecha_desde_nombre)
riesgos_validos['fecha_documento_texto'] = riesgos_validos['fecha_documento'].dt.strftime('%Y-%m-%d')

fechas_por_documento = (
    riesgos_validos[['source_filename', 'fecha_documento_texto']]
    .drop_duplicates()
    .sort_values(['fecha_documento_texto', 'source_filename'], na_position='last')
)

display(fechas_por_documento)

print('Riesgos con fecha inferida:', riesgos_validos['fecha_documento'].notna().sum())
print('Riesgos sin fecha inferida:', riesgos_validos['fecha_documento'].isna().sum())

## 7. Identificar documentos sin fecha

In [ ]:
docs_sin_fecha = (
    riesgos_validos[riesgos_validos['fecha_documento'].isna()]
    [['source_filename']]
    .drop_duplicates()
    .sort_values('source_filename')
)

print('Documentos sin fecha inferida:', len(docs_sin_fecha))
display(docs_sin_fecha)

## 8. Calcular score y prioridad del riesgo

In [ ]:
severity_score_map = {'Baja': 1, 'Media': 2, 'Alta': 3}
probability_score_map = {'Baja': 1, 'Media': 2, 'Alta': 3}

riesgos_validos['severity_score'] = riesgos_validos['severity'].map(severity_score_map).fillna(2).astype(int)
riesgos_validos['probability_score'] = riesgos_validos['probability'].map(probability_score_map).fillna(2).astype(int)
riesgos_validos['risk_score'] = riesgos_validos['severity_score'] * riesgos_validos['probability_score']

def clasificar_prioridad(score):
    if score >= 9:
        return 'Crítico'
    elif score >= 6:
        return 'Alto'
    elif score >= 3:
        return 'Medio'
    return 'Bajo'

riesgos_validos['risk_priority'] = riesgos_validos['risk_score'].apply(clasificar_prioridad)

display(
    riesgos_validos[['risk_id', 'risk_name', 'risk_category', 'severity', 'probability', 'risk_score', 'risk_priority']]
    .head(10)
)

## 9. Construir timeline de riesgos

In [ ]:
timeline_cols = [
    'risk_id', 'risk_name', 'risk_category', 'severity', 'probability',
    'risk_score', 'risk_priority', 'risk_description', 'evidence',
    'recommended_action', 'source_doc_id', 'source_filename', 'source_page',
    'source_chunk_id', 'fecha_documento', 'fecha_documento_texto'
]

available_cols = [c for c in timeline_cols if c in riesgos_validos.columns]

timeline_riesgos = riesgos_validos[available_cols].copy()

timeline_riesgos = timeline_riesgos.sort_values(
    by=['fecha_documento', 'risk_score', 'risk_category'],
    ascending=[True, False, True],
    na_position='last'
).reset_index(drop=True)

display(timeline_riesgos.head(20))
print('Eventos de riesgo en timeline:', len(timeline_riesgos))

## 10. Resumen para radar de riesgos

In [ ]:
radar_riesgos_resumen = (
    timeline_riesgos.groupby('risk_category')
    .agg(
        cantidad_riesgos=('risk_id', 'count'),
        score_promedio=('risk_score', 'mean'),
        score_maximo=('risk_score', 'max')
    )
    .reset_index()
    .sort_values(['score_maximo', 'cantidad_riesgos'], ascending=False)
)

display(radar_riesgos_resumen)

## 11. Riesgos priorizados

In [ ]:
riesgos_priorizados = timeline_riesgos.sort_values(
    by=['risk_score', 'risk_category', 'fecha_documento'],
    ascending=[False, True, True],
    na_position='last'
).reset_index(drop=True)

display(
    riesgos_priorizados[[
        'risk_id', 'risk_name', 'risk_category', 'severity', 'probability',
        'risk_score', 'risk_priority', 'fecha_documento_texto',
        'source_filename', 'source_chunk_id'
    ]].head(20)
)

## 12. Visualizaciones iniciales

In [ ]:
plot_df = radar_riesgos_resumen.sort_values('cantidad_riesgos', ascending=True)

plt.figure(figsize=(9, 5))
plt.barh(plot_df['risk_category'], plot_df['cantidad_riesgos'])
plt.title('Cantidad de riesgos por categoría')
plt.xlabel('Cantidad de riesgos')
plt.ylabel('Categoría')
plt.show()

priority_counts = (
    timeline_riesgos.groupby('risk_priority')
    .size()
    .reset_index(name='cantidad')
)

plt.figure(figsize=(7, 4))
plt.bar(priority_counts['risk_priority'], priority_counts['cantidad'])
plt.title('Cantidad de riesgos por prioridad')
plt.xlabel('Prioridad')
plt.ylabel('Cantidad de riesgos')
plt.show()

## 13. Timeline interactivo

In [ ]:
timeline_plot_df = timeline_riesgos[timeline_riesgos['fecha_documento'].notna()].copy()

if len(timeline_plot_df) > 0:
    fig = px.scatter(
        timeline_plot_df,
        x='fecha_documento',
        y='risk_category',
        size='risk_score',
        hover_data=['risk_id', 'risk_name', 'severity', 'probability', 'source_filename'],
        title='Timeline de riesgos documentales'
    )
    fig.show()
else:
    print('No hay fechas inferidas suficientes para construir timeline interactivo.')

## 14. Plantilla para completar fechas manualmente

Si algún documento queda sin fecha o si la fecha inferida corresponde solamente al año, puedes completar manualmente `fecha_documento_manual`.

In [ ]:
timeline_manual_dates = timeline_riesgos.copy()
timeline_manual_dates['fecha_documento_manual'] = ''
timeline_manual_dates['comentario_fecha'] = ''

display(
    timeline_manual_dates[[
        'risk_id', 'risk_name', 'source_filename', 'fecha_documento_texto',
        'fecha_documento_manual', 'comentario_fecha'
    ]].head(10)
)

## 15. Guardar artefactos

In [ ]:
timeline_riesgos.to_csv('timeline_riesgos.csv', index=False, encoding='utf-8-sig')
timeline_riesgos.to_excel('timeline_riesgos.xlsx', index=False)
radar_riesgos_resumen.to_csv('radar_riesgos_resumen.csv', index=False, encoding='utf-8-sig')
riesgos_priorizados.to_csv('riesgos_priorizados.csv', index=False, encoding='utf-8-sig')
riesgos_priorizados.to_excel('riesgos_priorizados.xlsx', index=False)
timeline_manual_dates.to_excel('timeline_riesgos_completar_fechas.xlsx', index=False)
docs_sin_fecha.to_csv('documentos_sin_fecha_inferida.csv', index=False, encoding='utf-8-sig')

print('Archivos generados:')
print('- timeline_riesgos.csv')
print('- timeline_riesgos.xlsx')
print('- radar_riesgos_resumen.csv')
print('- riesgos_priorizados.csv')
print('- riesgos_priorizados.xlsx')
print('- timeline_riesgos_completar_fechas.xlsx')
print('- documentos_sin_fecha_inferida.csv')

## 16. Descargar artefactos

In [ ]:
files.download('timeline_riesgos.csv')
files.download('timeline_riesgos.xlsx')
files.download('radar_riesgos_resumen.csv')
files.download('riesgos_priorizados.csv')
files.download('riesgos_priorizados.xlsx')
files.download('timeline_riesgos_completar_fechas.xlsx')
files.download('documentos_sin_fecha_inferida.csv')